In [ ]:

#output code snippet

    # Extract the final routes and compute summary metrics from the solved model.
    optimized_routes = []
    total_distance_km = 0

    # Iterate through each vehicle to reconstruct its actual route from the OR-Tools solution.
    for vehicle_id in range(data["num_vehicles"]):
        index = routing.Start(vehicle_id)
        route_stops = []
        route_distance = 0

        # Walk the route from the vehicle start until the end node is reached.
        while not routing.IsEnd(index):
            node_idx = manager.IndexToNode(index)
            node_data = data['nodes'][node_idx]
            time_var = time_dimension.CumulVar(index)

            # Record stop-related time data so the solution can be audited or displayed in a report.
            route_stops.append({
                "stop_id": node_data.get('id', node_idx),
                "name": node_data.get('name', f"Stop {node_idx}"),
                "arrival_min": solution.Min(time_var),
                "departure_min": solution.Min(time_var) + node_data.get('service_time', 0)
            })

            # Advance to the next stop in the route and accumulate the arc cost for this segment.
            previous_index = index
            index = solution.Value(routing.NextVar(index))
            from_node = manager.IndexToNode(previous_index)
            to_node = manager.IndexToNode(index)
            route_distance += data["distance_matrix"][from_node][to_node]

        # Save the full route for this vehicle.
        optimized_routes.append({
            "vehicle_id": vehicle_id + 1,
            "stops": route_stops,
            "distance_km": route_distance
        })
        total_distance_km += route_distance
